In [72]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,MatchID,Season,MatchWeek,HomeTeam,AwayTeam,FullTimeResult,Goals,Shots,Corners,YellowCards,RedCards,TotalCards,TeamStyles,chaos_label
0,2000-2001_Charlton_Man City,2000-2001,1,Charlton,Man City,H,4,25.0,12.0,3.0,0.0,3.0,"RiskTaker, DirectPlay",0
1,2000-2001_Chelsea_West Ham,2000-2001,1,Chelsea,West Ham,H,6,29.0,14.0,3.0,0.0,3.0,"RiskTaker, HighPressure, ChaosInducer",1
2,2000-2001_Coventry_Middlesbrough,2000-2001,1,Coventry,Middlesbrough,A,4,22.0,12.0,8.0,1.0,10.0,"AggressiveTackler, ChaosInducer, LowTempoContr...",1
3,2000-2001_Derby_Southampton,2000-2001,1,Derby,Southampton,D,4,19.0,13.0,2.0,0.0,2.0,"Disciplined, LowTempoController",0
4,2000-2001_Leeds_Everton,2000-2001,1,Leeds,Everton,H,2,29.0,10.0,4.0,0.0,4.0,"HighPressure, Opportunistic",0


In [73]:
df_train.columns

Index(['MatchID', 'Season', 'MatchWeek', 'HomeTeam', 'AwayTeam',
       'FullTimeResult', 'Goals', 'Shots', 'Corners', 'YellowCards',
       'RedCards', 'TotalCards', 'TeamStyles', 'chaos_label'],
      dtype='object')

In [74]:
#subtaskuri

subtask1 = len(df_test[(df_test.HomeTeam == 'Chelsea') | (df_test.AwayTeam == 'Chelsea')])

df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": 1,
    "answer": [subtask1]
})

df_1.head()

,subtaskID,datapointID,answer
0,1,1,150


In [75]:
aggresive_styles = {"AggressiveTackler", "RiskTaker", "HighPressure", "ChaosInducer"}

subtask2 = df_test["TeamStyles"].apply(
    lambda styles: sum(s in aggresive_styles for s in styles.split(", ")) / len(styles.split(", "))
)

df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.MatchID,
    "answer": subtask2.round(decimals=3)
})

df_2.head()

,subtaskID,datapointID,answer
0,2,2021-2022_Man City_Arsenal,0.75
1,2,2021-2022_Aston Villa_Brentford,1.00
2,2,2021-2022_Brighton_Everton,0.00
3,2,2021-2022_Newcastle_Southampton,1.00
4,2,2021-2022_Norwich_Leicester,0.50


In [76]:
#preprocessing (adaugam feature-ul, label enc si multilabelbinarizer)

df_train["AggresiveStyleCount"] = df_train["TeamStyles"].apply(lambda styles: sum(s in aggresive_styles for s in styles.split(", ")) / len(styles.split(", ")))
df_test["AggresiveStyleCount"] = df_test["TeamStyles"].apply(lambda styles: sum(s in aggresive_styles for s in styles.split(", ")) / len(styles.split(", ")))

from sklearn.preprocessing import LabelEncoder
to_enc = ['HomeTeam', 'AwayTeam', 'FullTimeResult']

for col in to_enc:
    enc = LabelEncoder()
    enc.fit(pd.concat([df_train[col], df_test[col]]))
    df_train[col] = enc.transform(df_train[col])
    df_test[col] = enc.transform(df_test[col])

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
styles_df = pd.DataFrame(
    mlb.fit_transform(df_train["TeamStyles"].str.split(", ")),
    columns=[f"style_{c}" for c in mlb.classes_], index=df_train.index)
df_train = pd.concat([df_train.drop(columns=["TeamStyles"]), styles_df], axis=1)

styles_df = pd.DataFrame(
    mlb.transform(df_test["TeamStyles"].str.split(", ")),
    columns=[f"style_{c}" for c in mlb.classes_], index=df_test.index)
df_test = pd.concat([df_test.drop(columns=["TeamStyles"]), styles_df], axis=1)

df_test.head()

,MatchID,Season,MatchWeek,HomeTeam,AwayTeam,FullTimeResult,Goals,Shots,Corners,YellowCards,...,TotalCards,AggresiveStyleCount,style_AggressiveTackler,style_ChaosInducer,style_DirectPlay,style_Disciplined,style_HighPressure,style_LowTempoController,style_Opportunistic,style_RiskTaker
0,2021-2022_Man City_Arsenal,2021-2022,3,26,0,2,5,26.0,14.0,3.0,...,5.0,0.75,0,1,0,0,1,1,0,1
1,2021-2022_Aston Villa_Brentford,2021-2022,3,1,8,1,2,16.0,5.0,6.0,...,6.0,1.00,1,1,0,0,0,0,0,0
2,2021-2022_Brighton_Everton,2021-2022,3,9,17,0,2,28.0,9.0,3.0,...,3.0,0.00,0,0,0,0,0,1,1,0
3,2021-2022_Newcastle_Southampton,2021-2022,3,29,36,1,4,32.0,8.0,6.0,...,6.0,1.00,1,1,0,0,1,0,0,1
4,2021-2022_Norwich_Leicester,2021-2022,3,30,23,0,3,23.0,18.0,3.0,...,3.0,0.50,0,0,0,0,1,1,0,0


In [77]:
#se poate si asa fara mlb

#all_styles = sorted({style for styles in df_train["TeamStyles"] for style in styles})
#for style in all_styles:
    #df[f"style_{style}"] = df["TeamStyles"].apply(lambda x: int(style in x))
#df = df.drop(columns=["TeamStyles"])

In [78]:
df_train.columns

Index(['MatchID', 'Season', 'MatchWeek', 'HomeTeam', 'AwayTeam',
       'FullTimeResult', 'Goals', 'Shots', 'Corners', 'YellowCards',
       'RedCards', 'TotalCards', 'chaos_label', 'AggresiveStyleCount',
       'style_AggressiveTackler', 'style_ChaosInducer', 'style_DirectPlay',
       'style_Disciplined', 'style_HighPressure', 'style_LowTempoController',
       'style_Opportunistic', 'style_RiskTaker'],
      dtype='object')

In [79]:
features = ['MatchWeek', 'HomeTeam', 'AwayTeam', 'FullTimeResult', 'Goals', 'Shots', 'Corners', 'YellowCards',
       'RedCards', 'TotalCards', 'style_AggressiveTackler', 'style_ChaosInducer', 'style_DirectPlay', 'style_Disciplined',
       'style_HighPressure', 'style_LowTempoController', 'style_Opportunistic', 'style_RiskTaker', 'AggresiveStyleCount']
target = 'chaos_label'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

X_final.head()

,MatchWeek,HomeTeam,AwayTeam,FullTimeResult,Goals,Shots,Corners,YellowCards,RedCards,TotalCards,style_AggressiveTackler,style_ChaosInducer,style_DirectPlay,style_Disciplined,style_HighPressure,style_LowTempoController,style_Opportunistic,style_RiskTaker,AggresiveStyleCount
0,3,26,0,2,5,26.0,14.0,3.0,1.0,5.0,0,1,0,0,1,1,0,1,0.75
1,3,1,8,1,2,16.0,5.0,6.0,0.0,6.0,1,1,0,0,0,0,0,0,1.00
2,3,9,17,0,2,28.0,9.0,3.0,0.0,3.0,0,0,0,0,0,1,1,0,0.00
3,3,29,36,1,4,32.0,8.0,6.0,0.0,6.0,1,1,0,0,1,0,0,1,1.00
4,3,30,23,0,3,23.0,18.0,3.0,0.0,3.0,0,0,0,0,1,1,0,0,0.50


In [97]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

model = RandomForestClassifier(n_estimators=250, max_depth=10, random_state=42)

cross_val_score(model, X, y, cv=5, scoring='f1_macro').mean()

0.9374421923074276

In [98]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [99]:
df_preds = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.MatchID,
    "answer": y_preds
})

df_preds.head()

,subtaskID,datapointID,answer
0,3,2021-2022_Man City_Arsenal,1
1,3,2021-2022_Aston Villa_Brentford,1
2,3,2021-2022_Brighton_Everton,0
3,3,2021-2022_Newcastle_Southampton,1
4,3,2021-2022_Norwich_Leicester,0


In [100]:
submission = pd.concat([df_1, df_2, df_preds])
submission

,subtaskID,datapointID,answer
0,1,1,150.00
0,2,2021-2022_Man City_Arsenal,0.75
1,2,2021-2022_Aston Villa_Brentford,1.00
2,2,2021-2022_Brighton_Everton,0.00
3,2,2021-2022_Newcastle_Southampton,1.00
...,...,...,...
1495,3,2024-2025_Newcastle_Everton,0.00
1496,3,2024-2025_Nott'm Forest_Chelsea,0.00
1497,3,2024-2025_Southampton_Arsenal,0.00
1498,3,2024-2025_Tottenham_Brighton,1.00


In [101]:
submission.to_csv("submission.csv", index=False)